<a id="grey-relational-analysis"></a>

$$
\text{灰色关联分析 (Grey Relational Analysis, GRA)} \\
\text{目的: 判断几个因素中, 哪个和目标的 "发展趋势" 最相似 (关联程度最高)} \\
\text{例如: GDP 的增长和第一, 二, 三产业中哪一个关系最密切?} \\ \quad \\
\text{和相关系数的区别: 相关系数需要较多样本并假设线性关系; 灰色关联只比较曲线形状的接近程度, 样本少时也能用} \\
\text{(灰色系统的思想: 信息少, 部分已知部分未知, 见灰色预测模型)} \\ \quad \\
\text{1. 参考序列 (母序列, Reference Sequence): } X_0 = (x_0(1), x_0(2), \cdots, x_0(n)) \\
\text{2. 比较序列 (子序列, Comparison Sequence): } X_i = (x_i(1), \cdots, x_i(n)), \ i = 1, \cdots, m
$$

相关模型: [灰色预测模型 GM(1,1)](gray_prediction_model.ipynb) · 数学基础: [皮尔逊相关系数](../../math_basics/statistics.ipynb#pearson)

<a id="gra-normalization"></a>

$$
\text{3. 无量纲化 (Dimensionless Processing): 各序列单位, 数量级不同, 先变成可以比较的形式} \\
\text{初值化 (Initial Value): } x_i'(k) = \frac{x_i(k)}{x_i(1)} \quad (\text{每个序列都从 1 开始, 比较 "相对于起点增长了多少"}) \\
\text{均值化 (Mean Value): } x_i'(k) = \frac{x_i(k)}{\bar{x}_i} \\
\text{两者都满足: 序列整体乘以常数 c, 结果不变 (} \frac{c x_i(k)}{c x_i(1)} = \frac{x_i(k)}{x_i(1)} \text{), 所以和单位无关} \\
\text{(做评价时, 也可以用 TOPSIS 里的正向化 + 归一化)}
$$

数学基础: [归一化与标准化](../../math_basics/statistics.ipynb#normalization)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

# 例子: 某地区 2017-2022 年 GDP 和三个产业的增加值 (亿元)
years = np.arange(2017, 2023)
gdp = np.array([1988, 2061, 2335, 2750, 3356, 3806])
industry = np.array([[386, 408, 422, 482, 511, 561],      # 第一产业
                     [839, 846, 960, 1258, 1577, 1893],   # 第二产业
                     [763, 808, 953, 1010, 1268, 1352]])  # 第三产业
names = ['primary', 'secondary', 'tertiary']

def initial_value(x):
    x = np.asarray(x, dtype=float)
    return x / x[..., :1]

def mean_value(x):
    x = np.asarray(x, dtype=float)
    return x / x.mean(axis=-1, keepdims=True)

X0 = initial_value(gdp)
Xi = initial_value(industry)
print(f'初值化后的参考序列 = {np.round(X0, 4)}')
print(f'初值化后的比较序列 = \n{np.round(Xi, 4)}')
print(f'换单位 (亿元 -> 万元) 后初值化结果不变? {np.allclose(initial_value(industry * 1e4), Xi)}')

plt.figure(figsize=(4, 3))
plt.plot(years, X0, 'k-o', linewidth=2, label='GDP (reference)')
for name, x in zip(names, Xi):
    plt.plot(years, x, '--o', markersize=3, label=name)
plt.ylabel('x(k) / x(1)')
plot_nicely()
plt.show()

<a id="grey-relational-coefficient"></a>

$$
\text{4. 灰色关联系数 (Grey Relational Coefficient): 在第 k 个时刻, 比较序列 i 和参考序列有多接近} \\
\Delta_i(k) = |x_0(k) - x_i(k)| \quad (\text{绝对差}) \\
\Delta_{\min} = \min_i \min_k \Delta_i(k), \quad \Delta_{\max} = \max_i \max_k \Delta_i(k) \quad (\text{两级最小差, 两级最大差, 在所有序列和所有时刻中取}) \\
\xi_i(k) = \frac{\Delta_{\min} + \rho \Delta_{\max}}{\Delta_i(k) + \rho \Delta_{\max}}, \quad \rho \in (0, 1) \text{ 为分辨系数 (Resolution Coefficient), 一般取 0.5} \\ \quad \\
\text{推导性质:} \\
\text{(1) 关于 } \Delta_i(k) \text{ 递减: 差越小, 关联系数越大} \\
\text{(2) 最大值: } \Delta_i(k) = \Delta_{\min} \text{ 时 } \xi = 1 \\
\text{(3) 最小值: } \Delta_i(k) = \Delta_{\max} \text{ 时 } \xi_{\min} = \frac{\Delta_{\min} + \rho \Delta_{\max}}{\Delta_{\max} + \rho \Delta_{\max}} \text{, 当 } \Delta_{\min} = 0 \text{ 时 } \xi_{\min} = \frac{\rho}{1 + \rho} \\
\text{所以 } \xi \in \left[\frac{\rho}{1 + \rho}, 1\right] \text{ (近似), } \rho = 0.5 \text{ 时为 } [\frac{1}{3}, 1] \\ \quad \\
\text{分辨系数 } \rho \text{ 的作用:} \\
\text{(a) } \rho \text{ 越小, } \xi \text{ 的取值范围越大, 不同序列之间的差别被放大, 分辨力越强} \\
\text{(b) 分子分母都加上 } \rho \Delta_{\max} \text{, 避免某个特别大的 } \Delta_{\max} \text{ (离群点) 让其他所有的关联系数都挤在一起} \\
\text{(c) 分子加上 } \Delta_{\min} \text{ 是为了让最接近的点正好等于 1}
$$

In [ ]:
def grey_relational_coefficient(X0, Xi, rho=0.5):
    delta = np.abs(Xi - X0)                    # m x n
    d_min, d_max = delta.min(), delta.max()   # 两级最小差 / 两级最大差
    return (d_min + rho * d_max) / (delta + rho * d_max), delta

xi, delta = grey_relational_coefficient(X0, Xi)
print(f'绝对差 Delta = \n{np.round(delta, 4)}')
print(f'关联系数 xi = \n{np.round(xi, 4)}')
print(f'xi 的范围 = [{xi.min():.4f}, {xi.max():.4f}], 理论下界 rho / (1 + rho) = {0.5 / 1.5:.4f} (Delta_min = {delta.min():.4f})')

# 独立验证: 用两重循环逐个计算
xi_loop = np.zeros_like(xi)
all_deltas = [abs(X0[k] - Xi[i, k]) for i in range(Xi.shape[0]) for k in range(Xi.shape[1])]
for i in range(Xi.shape[0]):
    for k in range(Xi.shape[1]):
        xi_loop[i, k] = (min(all_deltas) + 0.5 * max(all_deltas)) / (abs(X0[k] - Xi[i, k]) + 0.5 * max(all_deltas))
print(f'向量化 == 双重循环 ? {np.allclose(xi, xi_loop)}')

<a id="grey-relational-grade"></a>

$$
\text{5. 灰色关联度 (Grey Relational Grade): 把各时刻的关联系数平均, 得到整条曲线的关联程度} \\
r_i = \frac{1}{n} \sum_{k=1}^{n} \xi_i(k) \quad (\text{如果各时刻重要性不同, 也可以加权 } r_i = \sum_k w_k \xi_i(k)) \\
\text{按 } r_i \text{ 从大到小排序, 得到各因素对参考序列的影响程度 (关联序)} \\ \quad \\
\text{检查: 如果某个比较序列和参考序列形状完全相同 (成比例), 初值化后完全重合, } \Delta = 0, \ r = 1
$$

In [ ]:
r = xi.mean(axis=1)
for name, ri in zip(names, r):
    print(f'{name:<10}: 关联度 r = {ri:.4f}')
print(f'关联序: {[names[i] for i in np.argsort(-r)]}')

# 和皮尔逊相关系数对比: 6 个样本, 相关系数都很接近 1, 区分度低
# 两者结论也可能不同: Pearson 衡量 "线性程度", 灰色关联衡量初值化后两条曲线的距离 (增长幅度是否接近)
for name, x in zip(names, industry):
    print(f'{name:<10}: Pearson r = {np.corrcoef(gdp, x)[0, 1]:.4f}')

# 检查: 和 GDP 成比例的序列关联度为 1
X_with_copy = np.vstack([Xi, initial_value(0.3 * gdp)])
xi_copy, _ = grey_relational_coefficient(X0, X_with_copy)
print(f'加入一个 0.3 * GDP 的序列, 它的关联度 = {xi_copy[-1].mean():.4f}')

# 分辨系数的影响
rhos = np.linspace(0.05, 1, 40)
grades = np.array([grey_relational_coefficient(X0, Xi, rho)[0].mean(axis=1) for rho in rhos])
plt.figure(figsize=(4, 3))
for j, name in enumerate(names):
    plt.plot(rhos, grades[:, j], label=name)
plt.axvline(0.5, color='gray', linestyle='--', label='rho = 0.5')
plt.xlabel('resolution coefficient rho')
plt.ylabel('relational grade')
plot_nicely()
plt.show()
print(f'rho = 0.1 时关联度的极差 = {np.ptp(grades[np.argmin(np.abs(rhos - 0.1))]):.4f}, rho = 1 时 = {np.ptp(grades[-1]):.4f} (rho 越小区分度越大)')

<a id="gra-evaluation"></a>

$$
\text{6. 用灰色关联分析做综合评价: 参考序列取 "理想对象" (每个指标的最优值), 和 TOPSIS 的最优解 } Z^+ \text{ 一样} \\
\text{这时 "序列" 不再是时间, 而是指标: 对象 i 的指标向量 } (x_i(1), \cdots, x_i(m)) \text{ 作为比较序列} \\
\text{步骤: 正向化 → 归一化 (每列除以列均值或 min-max) → 参考序列 = 每列最大值 → 关联系数 → 关联度 (可加权) → 排序} \\
\text{关联度越大, 越接近理想对象, 评价越好}
$$

相关模型: [TOPSIS](../topsis.ipynb) · [熵权法](../entropy_weight_method.ipynb)

In [ ]:
# 5 个供应商, 指标: 质量 (极大), 价格 (极小), 交货准时率 (极大), 服务评分 (极大)
suppliers = ['S1', 'S2', 'S3', 'S4', 'S5']
data = np.array([[92, 120, 0.95, 8.1],
                 [88, 100, 0.90, 7.5],
                 [95, 135, 0.97, 8.8],
                 [85,  95, 0.85, 7.0],
                 [90, 110, 0.93, 9.0]], dtype=float)
data_forward = data.copy()
data_forward[:, 1] = data[:, 1].max() - data[:, 1]               # 价格: 极小型 -> 极大型
norm = (data_forward - data_forward.min(axis=0)) / np.ptp(data_forward, axis=0)   # min-max 到 [0, 1]

reference = norm.max(axis=0)                                     # 理想对象
xi_eval, _ = grey_relational_coefficient(reference, norm)
grade = xi_eval.mean(axis=1)

# TOPSIS (等权) 作对比
Z = norm
D_plus = np.linalg.norm(Z.max(axis=0) - Z, axis=1)
D_minus = np.linalg.norm(Z.min(axis=0) - Z, axis=1)
topsis_score = D_minus / (D_plus + D_minus)
for s, g, t in zip(suppliers, grade, topsis_score):
    print(f'{s}: 灰色关联度 = {g:.4f}, TOPSIS 得分 = {t:.4f}')
print(f'灰色关联排名: {[suppliers[i] for i in np.argsort(-grade)]}')
print(f'TOPSIS 排名:   {[suppliers[i] for i in np.argsort(-topsis_score)]}')